# Diffusion Models: From Noise to Data

Diffusion models generate data by learning to reverse a gradual noising process.
This notebook covers:
1. **Forward process**: progressively add noise
2. **Noise schedule**: variance schedule $\beta_t$
3. **Reverse process**: learn to denoise
4. **1D demonstration**: generate samples from a target distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
np.random.seed(42)
print('Setup complete.')

## 1. The Forward (Noising) Process

Starting from data $x_0$, we add Gaussian noise over $T$ steps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t I)$$

Thanks to the reparameterisation trick, we can jump directly to step $t$:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

where $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s=1}^{t} \alpha_s$.

In [ ]:
# Define noise schedule
T = 100  # number of diffusion steps
beta_start, beta_end = 1e-4, 0.02
betas = np.linspace(beta_start, beta_end, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(betas, 'b-')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel(r'$\beta_t$')
axes[0].set_title('Noise Schedule (linear)')

axes[1].plot(alpha_bar, 'r-')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel(r'$\bar{\alpha}_t$')
axes[1].set_title('Cumulative Signal Retention')
axes[1].axhline(0, ls='--', color='gray')
plt.tight_layout()
plt.show()

print(f'At t=0:  signal fraction = {alpha_bar[0]:.4f}')
print(f'At t={T//2}: signal fraction = {alpha_bar[T//2]:.4f}')
print(f'At t={T-1}: signal fraction = {alpha_bar[-1]:.4f} (nearly pure noise)')

In [ ]:
# Visualise the forward process on a 1D mixture of Gaussians
def sample_target(n):
    """Sample from a mixture of two Gaussians."""
    mix = np.random.rand(n) < 0.4
    return np.where(mix, np.random.normal(-2, 0.5, n), np.random.normal(2, 0.7, n))

x0 = sample_target(2000)

def forward_sample(x0, t, alpha_bar):
    """Sample x_t from the forward process."""
    eps = np.random.randn(*x0.shape)
    return np.sqrt(alpha_bar[t]) * x0 + np.sqrt(1 - alpha_bar[t]) * eps

timesteps_to_show = [0, 10, 30, 50, 80, 99]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(18, 3), sharey=True)

for ax, t in zip(axes, timesteps_to_show):
    xt = forward_sample(x0, t, alpha_bar)
    ax.hist(xt, bins=50, density=True, alpha=0.7, color='steelblue')
    ax.set_title(f't = {t}')
    ax.set_xlim(-6, 6)

axes[0].set_ylabel('Density')
plt.suptitle('Forward Process: Data -> Noise', y=1.02)
plt.tight_layout()
plt.show()

## 2. The Reverse (Denoising) Process

We learn a neural network $\varepsilon_\theta(x_t, t)$ that predicts the noise added at step $t$.

**Training objective (simplified):**
$$\mathcal{L} = \mathbb{E}_{t, x_0, \varepsilon}\left[\|\varepsilon - \varepsilon_\theta(x_t, t)\|^2\right]$$

**Sampling:** starting from $x_T \sim \mathcal{N}(0, I)$, iteratively denoise:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}} \varepsilon_\theta(x_t, t)\right) + \sigma_t z$$

In [ ]:
# Simple 1D denoiser: learn to predict noise using a small MLP (manual numpy implementation)

class SimpleMLP:
    """A tiny 2-layer MLP for 1D noise prediction."""
    def __init__(self, hidden=64):
        # Input: [x_t, t/T]  -> Output: predicted noise
        self.W1 = np.random.randn(2, hidden) * 0.1
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, hidden) * 0.1
        self.b2 = np.zeros(hidden)
        self.W3 = np.random.randn(hidden, 1) * 0.1
        self.b3 = np.zeros(1)
    
    def forward(self, x_t, t_norm):
        """Forward pass. x_t: (batch,), t_norm: (batch,) in [0,1]."""
        inp = np.stack([x_t, t_norm], axis=-1)  # (batch, 2)
        h = np.maximum(0, inp @ self.W1 + self.b1)  # ReLU
        h = np.maximum(0, h @ self.W2 + self.b2)
        out = h @ self.W3 + self.b3
        return out.flatten()
    
    def train_step(self, x0_batch, lr=1e-3):
        """One training step with numerical gradients (for simplicity)."""
        batch_size = len(x0_batch)
        t = np.random.randint(0, T, batch_size)
        t_norm = t / T
        eps = np.random.randn(batch_size)
        x_t = np.sqrt(alpha_bar[t]) * x0_batch + np.sqrt(1 - alpha_bar[t]) * eps
        
        eps_pred = self.forward(x_t, t_norm)
        loss = np.mean((eps - eps_pred)**2)
        
        # Numerical gradient update (simplified -- in practice use autograd)
        delta = 1e-4
        for param_name in ['W1', 'b1', 'W2', 'b2', 'W3', 'b3']:
            param = getattr(self, param_name)
            grad = np.zeros_like(param)
            it = np.nditer(param, flags=['multi_index'])
            # Only update a subset for speed
            indices = np.random.choice(param.size, min(10, param.size), replace=False)
            for idx in indices:
                multi_idx = np.unravel_index(idx, param.shape)
                old_val = param[multi_idx]
                param[multi_idx] = old_val + delta
                loss_plus = np.mean((eps - self.forward(x_t, t_norm))**2)
                param[multi_idx] = old_val - delta
                loss_minus = np.mean((eps - self.forward(x_t, t_norm))**2)
                param[multi_idx] = old_val
                grad[multi_idx] = (loss_plus - loss_minus) / (2 * delta)
            param -= lr * grad
        
        return loss

print('SimpleMLP denoiser defined (numpy-only, for demonstration).')

In [ ]:
# Train the denoiser
denoiser = SimpleMLP(hidden=32)
data = sample_target(5000)

losses = []
n_steps = 300
batch_size = 128

for step in range(n_steps):
    idx = np.random.choice(len(data), batch_size)
    loss = denoiser.train_step(data[idx], lr=5e-3)
    losses.append(loss)
    if (step + 1) % 100 == 0:
        print(f'Step {step+1}/{n_steps}, Loss: {loss:.4f}')

plt.plot(losses)
plt.xlabel('Training Step')
plt.ylabel('MSE Loss')
plt.title('Denoiser Training Loss')
plt.show()

In [ ]:
# Sampling: reverse process
def sample_reverse(denoiser, n_samples=1000):
    """Generate samples by reversing the diffusion process."""
    x = np.random.randn(n_samples)  # start from pure noise
    trajectory = [x.copy()]
    
    for t in range(T - 1, -1, -1):
        t_norm = np.full(n_samples, t / T)
        eps_pred = denoiser.forward(x, t_norm)
        
        # Reverse step
        coeff = betas[t] / np.sqrt(1 - alpha_bar[t])
        x = (x - coeff * eps_pred) / np.sqrt(alphas[t])
        
        if t > 0:
            noise = np.random.randn(n_samples) * np.sqrt(betas[t])
            x += noise
        
        if t % 20 == 0:
            trajectory.append(x.copy())
    
    return x, trajectory

generated, trajectory = sample_reverse(denoiser, n_samples=2000)

# Compare generated vs real
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(data, bins=60, density=True, alpha=0.6, label='Real data', color='steelblue')
axes[0].hist(generated, bins=60, density=True, alpha=0.6, label='Generated', color='coral')
axes[0].set_title('Real vs Generated Distribution')
axes[0].legend()

# Show reverse trajectory
n_show = min(6, len(trajectory))
for i, snap in enumerate(trajectory[:n_show]):
    axes[1].hist(snap, bins=40, density=True, alpha=0.3, label=f'Step {i}')
axes[1].set_title('Reverse Process Snapshots')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

print('Note: With a proper neural network and more training, the match would be much closer.')

## Key Takeaways

- **Forward process**: gradually destroy data by adding noise according to a schedule.
- **Reverse process**: a neural network learns to denoise, step by step.
- The training objective is simple: **predict the noise** $\varepsilon$ that was added.
- **Noise schedule** controls the trade-off between generation quality and speed.
- Extensions: DDPM, DDIM (faster sampling), classifier-free guidance, latent diffusion (Stable Diffusion).
- The connection to **score matching**: $\varepsilon_\theta \approx -\sqrt{1 - \bar\alpha_t}\, \nabla_x \log p(x_t)$.